# 03 · Capa Gold — Esquema estrella con SCD

**IDY1102 · EP3 · DataRetail 360 · Proceso Ventas**

| Campo | Detalle |
|---|---|
| **Capa Medallion** | Gold (modelo dimensional listo para análisis) |
| **Etapa ETL** | LOAD (sección 3 de la plantilla 2.2.3) |
| **IL** | IL 2.1 (modelo + capa semántica) · IL 2.2 (SCD 1, 2 y 3) · IL 2.4 (almacenamiento) |
| **Entrada** | Tablas `silver_*` de todos los lotes |
| **Salida** | `dim_*`, `fact_ventas`, vistas `vw_*` y diccionario `semantica_metricas` |

**Estrategia de carga: reconstrucción ordenada.** Gold se reconstruye en cada ejecución a partir de Silver, aplicando los lotes **en orden** (1, 2, ...). Como Silver conserva cada lote, el historial SCD2 se vuelve a generar igual cada vez. Resultado: ejecutar el notebook dos veces produce exactamente el mismo Gold (**idempotencia**), sin perder historial.

## 0. Configuración

In [1]:
import os
import sys
import hashlib
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import text

RAIZ = Path.cwd()
while not (RAIZ / "src" / "config.py").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.append(str(RAIZ / "src"))

import config
import utils

logger = utils.configurar_logging("gold")
motor = utils.obtener_motor()
load_dotenv(RAIZ / ".env")

LOTES = pd.read_sql("SELECT DISTINCT _id_lote FROM silver_ventas ORDER BY _id_lote", motor)["_id_lote"].tolist()
logger.info(f"=== Inicio capa Gold | lotes disponibles en Silver: {LOTES} ===")

2026-10-26 04:44:07 | INFO    | gold | === Inicio capa Gold | lotes disponibles en Silver: [1, 2] ===


## 1. Estructura de las tablas (DDL)

**DDL** (*Data Definition Language*) son las instrucciones SQL que crean tablas. Se definen explícitamente para declarar:
- **Llaves primarias** (`PRIMARY KEY`): identifican cada fila; no se pueden repetir.
- **Llaves foráneas** (`FOREIGN KEY ... REFERENCES`): cada venta debe apuntar a filas que existen en las dimensiones. La base de datos lo verifica sola (integridad referencial).

Es el diagrama de Draw.io convertido en tablas reales.

In [2]:
DDL = """
CREATE TABLE IF NOT EXISTS dim_tiempo (
    sk_tiempo INTEGER PRIMARY KEY, fecha TEXT NOT NULL UNIQUE, dia INTEGER, nombre_dia TEXT,
    es_fin_de_semana INTEGER, semana INTEGER, mes INTEGER, nombre_mes TEXT,
    trimestre INTEGER, anio INTEGER, es_feriado INTEGER);

CREATE TABLE IF NOT EXISTS dim_canal (
    sk_canal INTEGER PRIMARY KEY, cod_canal TEXT NOT NULL UNIQUE, nombre_canal TEXT, tipo_canal TEXT);

CREATE TABLE IF NOT EXISTS dim_sucursal (
    sk_sucursal INTEGER PRIMARY KEY, cod_sucursal TEXT NOT NULL UNIQUE, nombre_sucursal TEXT,
    tipo TEXT, comuna TEXT, region TEXT, zona TEXT, latitud REAL, longitud REAL, fecha_apertura TEXT);

CREATE TABLE IF NOT EXISTS dim_campana (
    sk_campana INTEGER PRIMARY KEY, cod_campana TEXT NOT NULL UNIQUE, nombre_campana TEXT,
    tipo_promocion TEXT, medio TEXT, fecha_inicio TEXT, fecha_fin TEXT, presupuesto INTEGER);

CREATE TABLE IF NOT EXISTS dim_producto (
    sk_producto INTEGER PRIMARY KEY, cod_producto TEXT NOT NULL, nombre_producto TEXT, marca TEXT,
    subcategoria TEXT, categoria TEXT, precio_lista INTEGER, precio_lista_anterior INTEGER, estado TEXT,
    fecha_inicio_vigencia TEXT, fecha_fin_vigencia TEXT, es_actual INTEGER);

CREATE TABLE IF NOT EXISTS dim_cliente (
    sk_cliente INTEGER PRIMARY KEY, id_cliente_seud TEXT NOT NULL, segmento TEXT, tramo_edad TEXT,
    comuna TEXT, region TEXT, fecha_registro TEXT,
    fecha_inicio_vigencia TEXT, fecha_fin_vigencia TEXT, es_actual INTEGER);

CREATE TABLE IF NOT EXISTS fact_ventas (
    sk_tiempo   INTEGER NOT NULL REFERENCES dim_tiempo(sk_tiempo),
    sk_cliente  INTEGER NOT NULL REFERENCES dim_cliente(sk_cliente),
    sk_producto INTEGER NOT NULL REFERENCES dim_producto(sk_producto),
    sk_sucursal INTEGER NOT NULL REFERENCES dim_sucursal(sk_sucursal),
    sk_canal    INTEGER NOT NULL REFERENCES dim_canal(sk_canal),
    sk_campana  INTEGER NOT NULL REFERENCES dim_campana(sk_campana),
    origen TEXT NOT NULL, nro_documento TEXT NOT NULL, nro_linea INTEGER NOT NULL,
    unidades INTEGER, precio_unitario INTEGER, monto_bruto INTEGER, descuento INTEGER,
    monto_neto INTEGER, costo INTEGER, margen INTEGER, es_devolucion INTEGER, es_anulada INTEGER,
    _id_lote INTEGER, _archivo_origen TEXT, _fila_origen INTEGER,
    PRIMARY KEY (origen, nro_documento, nro_linea));
"""

with motor.begin() as conexion:
    for sentencia in DDL.split(";"):
        if sentencia.strip():
            conexion.execute(text(sentencia))
logger.info("Estructura Gold verificada (7 tablas con llaves primarias y foráneas)")

2026-10-26 04:44:07 | INFO    | gold | Estructura Gold verificada (7 tablas con llaves primarias y foráneas)


## 2. Leer Silver

In [3]:
def leer_silver(tabla: str) -> pd.DataFrame:
    return pd.read_sql(f"SELECT * FROM silver_{tabla} ORDER BY _id_lote", motor)

silver = {t: leer_silver(t) for t in ("canales", "sucursales", "campanas", "productos", "clientes", "ventas")}
for t, df in silver.items():
    logger.info(f"Leído silver_{t}: {len(df)} filas en {df['_id_lote'].nunique()} lote(s)")

2026-10-26 04:44:07 | INFO    | gold | Leído silver_canales: 3 filas en 1 lote(s)
2026-10-26 04:44:07 | INFO    | gold | Leído silver_sucursales: 10 filas en 1 lote(s)
2026-10-26 04:44:07 | INFO    | gold | Leído silver_campanas: 5 filas en 1 lote(s)
2026-10-26 04:44:07 | INFO    | gold | Leído silver_productos: 67 filas en 2 lote(s)
2026-10-26 04:44:07 | INFO    | gold | Leído silver_clientes: 830 filas en 2 lote(s)
2026-10-26 04:44:07 | INFO    | gold | Leído silver_ventas: 14672 filas en 2 lote(s)


## 3. Dimensión Tiempo

No viene de ninguna fuente: se **genera** con el calendario. La llave `sk_tiempo` tiene el formato `AAAAMMDD` (por ejemplo, 20260415), así se puede leer a simple vista.

In [4]:
DIAS = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto",
         "Septiembre", "Octubre", "Noviembre", "Diciembre"]

fechas = pd.date_range(f"{config.ANIO_CALENDARIO}-01-01", f"{config.ANIO_CALENDARIO}-12-31", freq="D")
dim_tiempo = pd.DataFrame({
    "sk_tiempo": fechas.strftime("%Y%m%d").astype(int),
    "fecha": fechas.strftime("%Y-%m-%d"),
    "dia": fechas.day,
    "nombre_dia": [DIAS[d] for d in fechas.dayofweek],
    "es_fin_de_semana": (fechas.dayofweek >= 5).astype(int),
    "semana": fechas.isocalendar().week.astype(int).values,
    "mes": fechas.month,
    "nombre_mes": [MESES[m - 1] for m in fechas.month],
    "trimestre": fechas.quarter,
    "anio": fechas.year,
    "es_feriado": fechas.strftime("%Y-%m-%d").isin(config.FERIADOS).astype(int),
})
logger.info(f"dim_tiempo generada: {len(dim_tiempo)} días, {dim_tiempo['es_feriado'].sum()} feriados")

2026-10-26 04:44:07 | INFO    | gold | dim_tiempo generada: 365 días, 16 feriados


C:\Users\Isabe\AppData\Local\Temp\ipykernel_20396\3763833043.py:10: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  "nombre_dia": [DIAS[d] for d in fechas.dayofweek],
C:\Users\Isabe\AppData\Local\Temp\ipykernel_20396\3763833043.py:11: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  "es_fin_de_semana": (fechas.dayofweek >= 5).astype(int),


## 4. Dimensiones con SCD tipo 1: Canal, Sucursal y Campaña

**SCD1 = sobrescribir.** Si un atributo cambia, se reemplaza el valor anterior; no se guarda historial.

La función `aplicar_scd1` compara lo que llega de Silver con la dimensión:
- Si la llave natural **no existe** → se **inserta** con una llave sustituta nueva.
- Si **existe y algo cambió** → se **actualiza** en el mismo lugar.

In [5]:
def aplicar_scd1(dim: pd.DataFrame, entrada: pd.DataFrame, nk: str, sk: str, atributos: list) -> pd.DataFrame:
    """Aplica SCD tipo 1: inserta lo nuevo y sobrescribe lo que cambió."""
    dim = dim.copy()
    actualizados = 0
    for _, fila in entrada.iterrows():
        existe = dim[nk] == fila[nk]
        if existe.any():
            if (dim.loc[existe, atributos].values[0] != fila[atributos].values).any():
                dim.loc[existe, atributos] = fila[atributos].values
                actualizados += 1
        else:
            nueva = {sk: int(dim[sk].max()) + 1 if len(dim) else 1, nk: fila[nk]}
            nueva.update({a: fila[a] for a in atributos})
            dim = pd.concat([dim, pd.DataFrame([nueva])], ignore_index=True)
    return dim, actualizados


def construir_scd1(tabla: str, nk: str, sk: str, atributos: list) -> pd.DataFrame:
    """Recorre los lotes en orden aplicando SCD1."""
    dim = pd.DataFrame(columns=[sk, nk] + atributos)
    for lote in LOTES:
        entrada = silver[tabla][silver[tabla]["_id_lote"] == lote]
        if len(entrada) == 0:
            continue
        antes = len(dim)
        dim, actualizados = aplicar_scd1(dim, entrada, nk, sk, atributos)
        logger.info(f"[SCD1] dim de {tabla} | lote {lote}: {len(dim) - antes} insertadas, {actualizados} actualizadas")
    return dim


dim_canal = construir_scd1("canales", "cod_canal", "sk_canal", ["nombre_canal", "tipo_canal"])
dim_sucursal = construir_scd1("sucursales", "cod_sucursal", "sk_sucursal",
                              ["nombre_sucursal", "tipo", "comuna", "region", "zona",
                               "latitud", "longitud", "fecha_apertura"])
dim_campana = construir_scd1("campanas", "cod_campana", "sk_campana",
                             ["nombre_campana", "tipo_promocion", "medio", "fecha_inicio",
                              "fecha_fin", "presupuesto"])

# Fila especial sk = 0: ventas sin campaña (evita llaves foráneas nulas)
sin_campana = pd.DataFrame([{"sk_campana": 0, "cod_campana": "SIN_CAMPANA", "nombre_campana": "Sin campaña",
                             "tipo_promocion": "No aplica", "medio": "No aplica", "fecha_inicio": None,
                             "fecha_fin": None, "presupuesto": 0}])
dim_campana = pd.concat([sin_campana, dim_campana], ignore_index=True)

2026-10-26 04:44:07 | INFO    | gold | [SCD1] dim de canales | lote 1: 3 insertadas, 0 actualizadas
2026-10-26 04:44:08 | INFO    | gold | [SCD1] dim de sucursales | lote 1: 10 insertadas, 0 actualizadas
2026-10-26 04:44:08 | INFO    | gold | [SCD1] dim de campanas | lote 1: 5 insertadas, 0 actualizadas


## 5. Dimensiones con historial: Producto y Cliente (SCD 1, 2 y 3)

Una misma dimensión combina los tres tipos, según lo decidido en el diseño:

| Dimensión | SCD1 (sobrescribir) | SCD2 (nueva versión) | SCD3 (valor anterior) |
|---|---|---|---|
| Producto | nombre, marca, estado | subcategoría, categoría | precio_lista → `precio_lista_anterior` |
| Cliente | tramo de edad, fecha de registro | segmento, comuna, región | — |

**Cómo funciona SCD2 en `aplicar_scd`:** cuando cambia un atributo SCD2,
1. la versión actual se **cierra**: `fecha_fin_vigencia` = día anterior al inicio del lote, `es_actual = 0`;
2. se **inserta** una versión nueva con una **llave sustituta nueva**, vigente desde el inicio del lote (`es_actual = 1`, fin `9999-12-31`).

Así, las ventas antiguas siguen apuntando a la versión antigua y las nuevas a la nueva.

In [6]:
def aplicar_scd(dim: pd.DataFrame, entrada: pd.DataFrame, nk: str, sk: str, lote: int,
                scd1: list, scd2: list, scd3: dict) -> tuple:
    """Aplica SCD 1, 2 y 3 sobre una dimensión para un lote. Devuelve (dim, conteos)."""
    dim = dim.copy()
    conteo = {"insertadas": 0, "scd1": 0, "scd2": 0, "scd3": 0}
    inicio_lote = config.FECHA_INICIO_POR_LOTE[lote]
    fin_anterior = (pd.Timestamp(inicio_lote) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    primera_carga = len(dim) == 0
    siguiente_sk = int(dim[sk].max()) + 1 if len(dim) else 1
    nuevas_filas = []

    for _, fila in entrada.iterrows():
        de_esta_llave = dim[nk] == fila[nk]
        actual = de_esta_llave & (dim["es_actual"] == 1)

        if not actual.any():                      # --- llave nueva: insertar
            nueva = {c: fila[c] for c in scd1 + scd2 + list(scd3)}
            nueva.update({sk: siguiente_sk, nk: fila[nk], "es_actual": 1,
                          "fecha_inicio_vigencia": config.FECHA_INICIO_HISTORICA if primera_carga else inicio_lote,
                          "fecha_fin_vigencia": config.FECHA_FIN_ABIERTA})
            nueva.update({anterior: None for anterior in scd3.values()})
            nuevas_filas.append(nueva)
            siguiente_sk += 1
            conteo["insertadas"] += 1
            continue

        vigente = dim.loc[actual].iloc[0]

        # SCD1: sobrescribir en TODAS las versiones de esa llave
        for col in scd1:
            if vigente[col] != fila[col]:
                dim.loc[de_esta_llave, col] = fila[col]
                conteo["scd1"] += 1

        # SCD3: guardar el valor anterior en su columna y poner el nuevo
        for col, col_anterior in scd3.items():
            if vigente[col] != fila[col]:
                dim.loc[de_esta_llave, col_anterior] = vigente[col]
                dim.loc[de_esta_llave, col] = fila[col]
                conteo["scd3"] += 1

        # SCD2: si cambió algún atributo histórico, cerrar la versión y crear una nueva
        if any(vigente[col] != fila[col] for col in scd2):
            dim.loc[actual, "fecha_fin_vigencia"] = fin_anterior
            dim.loc[actual, "es_actual"] = 0
            nueva = dim.loc[actual].iloc[0].to_dict()          # copia con SCD1/SCD3 ya aplicados
            nueva.update({c: fila[c] for c in scd2})
            nueva.update({sk: siguiente_sk, "es_actual": 1,
                          "fecha_inicio_vigencia": inicio_lote,
                          "fecha_fin_vigencia": config.FECHA_FIN_ABIERTA})
            nuevas_filas.append(nueva)
            siguiente_sk += 1
            conteo["scd2"] += 1

    if nuevas_filas:
        dim = pd.concat([dim, pd.DataFrame(nuevas_filas)], ignore_index=True)
    return dim, conteo


def construir_dim_historica(tabla, nk, sk, scd1, scd2, scd3) -> pd.DataFrame:
    columnas = [sk, nk] + scd1 + scd2 + list(scd3) + list(scd3.values()) + \
               ["fecha_inicio_vigencia", "fecha_fin_vigencia", "es_actual"]
    dim = pd.DataFrame(columns=columnas)
    for lote in LOTES:
        entrada = silver[tabla][silver[tabla]["_id_lote"] == lote]
        dim, conteo = aplicar_scd(dim, entrada, nk, sk, lote, scd1, scd2, scd3)
        logger.info(f"[SCD] dim de {tabla} | lote {lote}: {conteo}")
    return dim[columnas]


dim_producto = construir_dim_historica(
    "productos", "cod_producto", "sk_producto",
    scd1=["nombre_producto", "marca", "estado"],
    scd2=["subcategoria", "categoria"],
    scd3={"precio_lista": "precio_lista_anterior"})

dim_cliente = construir_dim_historica(
    "clientes", "id_cliente_seud", "sk_cliente",
    scd1=["tramo_edad", "fecha_registro"],
    scd2=["segmento", "comuna", "region"],
    scd3={})

# Fila especial sk = 0: compras en tienda sin cliente identificado
no_identificado = pd.DataFrame([{"sk_cliente": 0, "id_cliente_seud": "NO_IDENTIFICADO",
                                 "tramo_edad": "No aplica", "fecha_registro": None, "segmento": "No identificado",
                                 "comuna": "No aplica", "region": "No aplica",
                                 "fecha_inicio_vigencia": config.FECHA_INICIO_HISTORICA,
                                 "fecha_fin_vigencia": config.FECHA_FIN_ABIERTA, "es_actual": 1}])
dim_cliente = pd.concat([no_identificado, dim_cliente], ignore_index=True)
logger.info(f"dim_producto: {len(dim_producto)} versiones de {dim_producto['cod_producto'].nunique()} productos")
logger.info(f"dim_cliente: {len(dim_cliente)} versiones de {dim_cliente['id_cliente_seud'].nunique()} clientes")

2026-10-26 04:44:08 | INFO    | gold | [SCD] dim de productos | lote 1: {'insertadas': 32, 'scd1': 0, 'scd2': 0, 'scd3': 0}
2026-10-26 04:44:08 | INFO    | gold | [SCD] dim de productos | lote 2: {'insertadas': 3, 'scd1': 1, 'scd2': 2, 'scd3': 4}
2026-10-26 04:44:08 | INFO    | gold | [SCD] dim de clientes | lote 1: {'insertadas': 400, 'scd1': 0, 'scd2': 0, 'scd3': 0}
2026-10-26 04:44:08 | INFO    | gold | [SCD] dim de clientes | lote 2: {'insertadas': 30, 'scd1': 21, 'scd2': 35, 'scd3': 0}
2026-10-26 04:44:08 | INFO    | gold | dim_producto: 37 versiones de 35 productos
2026-10-26 04:44:08 | INFO    | gold | dim_cliente: 466 versiones de 431 clientes


## 6. Tabla de hechos: Fact_Ventas

Para cada venta se busca la **llave sustituta** de cada dimensión (*lookup*):
- Tiempo, canal, sucursal y campaña: búsqueda directa por código.
- **Producto y cliente: búsqueda "en el tiempo"** (*point-in-time*). Se elige la versión cuya vigencia contiene la fecha de la venta. Es lo que hace funcionar el SCD2: una venta de marzo de un cliente que se mudó en julio queda con la versión de marzo.

In [7]:
v = silver["ventas"].copy()
filas_silver = len(v)

v["sk_tiempo"] = v["fecha"].str.replace("-", "").astype(int)
v["sk_canal"] = v["cod_canal"].map(dict(zip(dim_canal["cod_canal"], dim_canal["sk_canal"])))
v["sk_sucursal"] = v["cod_sucursal"].map(dict(zip(dim_sucursal["cod_sucursal"], dim_sucursal["sk_sucursal"])))
v["sk_campana"] = v["cod_campana"].map(dict(zip(dim_campana["cod_campana"], dim_campana["sk_campana"]))).fillna(0)


def buscar_version(ventas, dim, nk_ventas, nk_dim, sk):
    """Une cada venta con la versión de la dimensión vigente en la fecha de la venta."""
    versiones = dim[[sk, nk_dim, "fecha_inicio_vigencia", "fecha_fin_vigencia"]]
    unido = ventas.merge(versiones, left_on=nk_ventas, right_on=nk_dim, how="left")
    vigente = (unido["fecha"] >= unido["fecha_inicio_vigencia"]) & (unido["fecha"] <= unido["fecha_fin_vigencia"])
    return unido[vigente].drop(columns=[nk_dim, "fecha_inicio_vigencia", "fecha_fin_vigencia"])


v = buscar_version(v, dim_producto, "cod_producto", "cod_producto", "sk_producto")
con_cliente = v[v["id_cliente_seud"].notna()]
sin_cliente = v[v["id_cliente_seud"].isna()].assign(sk_cliente=0)
v = pd.concat([buscar_version(con_cliente, dim_cliente, "id_cliente_seud", "id_cliente_seud", "sk_cliente"),
               sin_cliente], ignore_index=True)

assert len(v) == filas_silver, f"El lookup perdió o duplicó ventas: {filas_silver} -> {len(v)}"

fact_ventas = v.rename(columns={"cantidad": "unidades"})[[
    "sk_tiempo", "sk_cliente", "sk_producto", "sk_sucursal", "sk_canal", "sk_campana",
    "origen", "nro_documento", "nro_linea", "unidades", "precio_unitario", "monto_bruto",
    "descuento", "monto_neto", "costo", "margen", "es_devolucion", "es_anulada",
    "_id_lote", "_archivo_origen", "_fila_origen"]]
fact_ventas = fact_ventas.astype({c: "int64" for c in ["sk_tiempo", "sk_cliente", "sk_producto",
                                                       "sk_sucursal", "sk_canal", "sk_campana"]})
logger.info(f"fact_ventas preparada: {len(fact_ventas)} filas")

2026-10-26 04:44:08 | INFO    | gold | fact_ventas preparada: 14672 filas


## 7. LOAD — cargar Gold

Orden obligatorio por las llaves foráneas (como indica la plantilla 2.2.3):
1. Borrar **primero los hechos** y luego las dimensiones.
2. Insertar **primero las dimensiones** y luego los hechos.

Todo en una sola transacción: si algo falla, Gold queda como estaba.

In [8]:
dimensiones = {"dim_tiempo": dim_tiempo, "dim_canal": dim_canal, "dim_sucursal": dim_sucursal,
               "dim_campana": dim_campana, "dim_producto": dim_producto, "dim_cliente": dim_cliente}

try:
    with motor.begin() as conexion:
        conexion.execute(text("DELETE FROM fact_ventas"))
        for tabla in dimensiones:
            conexion.execute(text(f"DELETE FROM {tabla}"))
        for tabla, df in dimensiones.items():
            df.to_sql(tabla, conexion, if_exists="append", index=False)
            utils.registrar_control(conexion, max(LOTES), "gold", tabla, "silver", len(df), "OK")
        fact_ventas.to_sql("fact_ventas", conexion, if_exists="append", index=False, chunksize=2000)
        utils.registrar_control(conexion, max(LOTES), "gold", "fact_ventas", "silver_ventas", len(fact_ventas), "OK")
except Exception as error:
    logger.error(f"Falló la carga de Gold; no se aplicó ningún cambio: {error}")
    raise

for tabla, df in {**dimensiones, "fact_ventas": fact_ventas}.items():
    logger.info(f"Cargado {tabla}: {len(df)} filas")

2026-10-26 04:44:08 | INFO    | gold | Cargado dim_tiempo: 365 filas
2026-10-26 04:44:08 | INFO    | gold | Cargado dim_canal: 3 filas
2026-10-26 04:44:08 | INFO    | gold | Cargado dim_sucursal: 10 filas
2026-10-26 04:44:08 | INFO    | gold | Cargado dim_campana: 6 filas
2026-10-26 04:44:08 | INFO    | gold | Cargado dim_producto: 37 filas
2026-10-26 04:44:08 | INFO    | gold | Cargado dim_cliente: 466 filas
2026-10-26 04:44:08 | INFO    | gold | Cargado fact_ventas: 14672 filas


## 8. Capa semántica para IA

Una IA que traduce preguntas a SQL (*text-to-SQL*) necesita **definiciones únicas y documentadas**. Si cada área calcula "venta neta" a su manera, la IA también lo hará mal (problema detectado en el EP1).

Se crean:
- **Vistas** (`vw_*`): consultas guardadas con las métricas ya calculadas según la definición oficial.
- **`semantica_metricas`**: un diccionario que explica cada métrica en lenguaje de negocio y con su fórmula SQL.

In [9]:
VISTAS = {
"vw_ventas": """
SELECT t.fecha, t.anio, t.mes, t.nombre_mes, t.trimestre, t.es_fin_de_semana, t.es_feriado,
       c.nombre_canal, c.tipo_canal, s.nombre_sucursal, s.comuna AS comuna_sucursal, s.region AS region_sucursal,
       s.zona, p.nombre_producto, p.marca, p.subcategoria, p.categoria,
       cl.segmento, cl.tramo_edad, cl.region AS region_cliente, cp.nombre_campana,
       f.origen, f.nro_documento, f.unidades, f.monto_bruto, f.descuento, f.monto_neto, f.costo, f.margen,
       f.es_devolucion
FROM fact_ventas f
JOIN dim_tiempo t    ON t.sk_tiempo = f.sk_tiempo
JOIN dim_canal c     ON c.sk_canal = f.sk_canal
JOIN dim_sucursal s  ON s.sk_sucursal = f.sk_sucursal
JOIN dim_producto p  ON p.sk_producto = f.sk_producto
JOIN dim_cliente cl  ON cl.sk_cliente = f.sk_cliente
JOIN dim_campana cp  ON cp.sk_campana = f.sk_campana
WHERE f.es_anulada = 0""",
"vw_kpi_mensual": """
SELECT anio, mes, nombre_mes, nombre_canal,
       SUM(monto_neto) AS venta_neta,
       SUM(margen) AS margen,
       SUM(unidades) AS unidades,
       COUNT(DISTINCT CASE WHEN es_devolucion = 0 THEN origen || nro_documento END) AS documentos_venta,
       ROUND(1.0 * SUM(CASE WHEN es_devolucion = 0 THEN monto_neto END)
             / COUNT(DISTINCT CASE WHEN es_devolucion = 0 THEN origen || nro_documento END), 0) AS ticket_promedio
FROM vw_ventas GROUP BY anio, mes, nombre_mes, nombre_canal""",
"vw_ventas_sucursal": """
SELECT zona, region_sucursal, nombre_sucursal,
       SUM(monto_neto) AS venta_neta, SUM(margen) AS margen,
       ROUND(100.0 * SUM(margen) / SUM(monto_neto), 1) AS margen_pct
FROM vw_ventas GROUP BY zona, region_sucursal, nombre_sucursal""",
"vw_roi_campana": """
SELECT cp.nombre_campana, cp.presupuesto, SUM(f.monto_neto) AS venta_neta, SUM(f.margen) AS margen,
       ROUND(1.0 * (SUM(f.margen) - cp.presupuesto) / cp.presupuesto, 2) AS roi
FROM fact_ventas f JOIN dim_campana cp ON cp.sk_campana = f.sk_campana
WHERE f.es_anulada = 0 AND cp.sk_campana <> 0
GROUP BY cp.nombre_campana, cp.presupuesto""",
}

semantica = pd.DataFrame([
    ("venta_neta", "Ingreso por ventas descontando descuentos y devoluciones. Excluye documentos anulados.",
     "SUM(monto_neto) WHERE es_anulada = 0", "vw_kpi_mensual", "Las devoluciones tienen monto negativo."),
    ("margen", "Venta neta menos el costo de los productos vendidos.",
     "SUM(monto_neto - costo) WHERE es_anulada = 0", "vw_kpi_mensual", "Costo según el maestro de productos del lote."),
    ("ticket_promedio", "Monto neto promedio por boleta o pedido (sin devoluciones ni anulados).",
     "SUM(monto_neto) / COUNT(DISTINCT origen || nro_documento)", "vw_kpi_mensual", "Se calcula, no se almacena."),
    ("unidades", "Cantidad de productos vendidos, neta de devoluciones.",
     "SUM(unidades) WHERE es_anulada = 0", "vw_kpi_mensual", ""),
    ("margen_pct", "Porcentaje de la venta neta que queda como margen.",
     "100 * SUM(margen) / SUM(monto_neto)", "vw_ventas_sucursal", ""),
    ("roi_campana", "Retorno de una campaña: margen generado menos presupuesto, dividido por el presupuesto.",
     "(SUM(margen) - presupuesto) / presupuesto", "vw_roi_campana", "Solo ventas asociadas a la campaña."),
], columns=["metrica", "definicion_negocio", "formula_sql", "vista", "notas"])

with motor.begin() as conexion:
    for nombre, sql in VISTAS.items():
        conexion.execute(text(f"DROP VIEW IF EXISTS {nombre}"))
    for nombre, sql in VISTAS.items():
        conexion.execute(text(f"CREATE VIEW {nombre} AS {sql}"))
    semantica.to_sql("semantica_metricas", conexion, if_exists="replace", index=False)
logger.info(f"Capa semántica creada: {len(VISTAS)} vistas y {len(semantica)} métricas documentadas")
semantica

2026-10-26 04:44:08 | INFO    | gold | Capa semántica creada: 4 vistas y 6 métricas documentadas


,metrica,definicion_negocio,formula_sql,vista,notas
0,venta_neta,Ingreso por ventas descontando descuentos y de...,SUM(monto_neto) WHERE es_anulada = 0,vw_kpi_mensual,Las devoluciones tienen monto negativo.
1,margen,Venta neta menos el costo de los productos ven...,SUM(monto_neto - costo) WHERE es_anulada = 0,vw_kpi_mensual,Costo según el maestro de productos del lote.
2,ticket_promedio,Monto neto promedio por boleta o pedido (sin d...,SUM(monto_neto) / COUNT(DISTINCT origen || nro...,vw_kpi_mensual,"Se calcula, no se almacena."
3,unidades,"Cantidad de productos vendidos, neta de devolu...",SUM(unidades) WHERE es_anulada = 0,vw_kpi_mensual,
4,margen_pct,Porcentaje de la venta neta que queda como mar...,100 * SUM(margen) / SUM(monto_neto),vw_ventas_sucursal,
5,roi_campana,Retorno de una campaña: margen generado menos ...,(SUM(margen) - presupuesto) / presupuesto,vw_roi_campana,Solo ventas asociadas a la campaña.


## 9. VALIDACIÓN post-carga

1. **Conteo:** Gold tiene la misma cantidad de ventas que Silver.
2. **Integridad referencial:** SQLite confirma que todas las llaves foráneas apuntan a filas existentes.
3. **Conciliación:** la venta neta total de Gold es igual a la de Silver (no se perdió ni duplicó dinero).
4. **SCD2 consistente:** cada producto y cliente tiene exactamente una versión actual.

In [10]:
with motor.connect() as conexion:
    filas_gold = conexion.execute(text("SELECT COUNT(*) FROM fact_ventas")).scalar()
    huerfanas = conexion.execute(text("PRAGMA foreign_key_check")).fetchall()
    neto_gold = conexion.execute(text("SELECT SUM(monto_neto) FROM fact_ventas")).scalar()
    neto_silver = conexion.execute(text("SELECT SUM(monto_neto) FROM silver_ventas")).scalar()

assert filas_gold == filas_silver, "Gold no tiene las mismas ventas que Silver"
assert len(huerfanas) == 0, f"Hay llaves foráneas rotas: {huerfanas[:5]}"
assert neto_gold == neto_silver, f"Conciliación fallida: Gold {neto_gold} vs Silver {neto_silver}"
for dim, nk in [(dim_producto, "cod_producto"), (dim_cliente, "id_cliente_seud")]:
    assert (dim[dim["es_actual"] == 1].groupby(nk).size() == 1).all(), f"{nk} con más de una versión actual"

logger.info(f"Validación OK | ventas: {filas_gold} | FK rotas: 0 | venta neta conciliada: ${neto_gold:,.0f}")

2026-10-26 04:44:08 | INFO    | gold | Validación OK | ventas: 14672 | FK rotas: 0 | venta neta conciliada: $2,848,690,374


## 10. Verificación del SCD con los cambios esperados

El generador anotó los 80 cambios del lote 2 en `data/cambios_esperados_lote2.csv`. Aquí se comprueba, uno por uno, que Gold los refleja correctamente.

In [11]:
SAL = os.getenv("SAL_SEUDONIMIZACION")
seud = lambda x: hashlib.sha256((SAL + x).encode("utf-8")).hexdigest()[:16]

cambios = pd.read_csv(RAIZ / "data" / "cambios_esperados_lote2.csv", dtype=str)
resultados = []
for _, c in cambios.iterrows():
    if c["maestro"] == "clientes":
        filas = dim_cliente[dim_cliente["id_cliente_seud"] == seud(c["registro"])]
    else:
        filas = dim_producto[dim_producto["cod_producto"] == c["registro"]]
    actual = filas[filas["es_actual"] == 1].iloc[0]

    if c["tratamiento"] == "SCD2":
        anterior = filas[filas["es_actual"] == 0]
        ok = (len(filas) == 2 and str(actual[c["atributo"]]) == c["valor_nuevo"]
              and str(anterior.iloc[0][c["atributo"]]) == c["valor_anterior"])
    elif c["tratamiento"] == "SCD3":
        ok = (len(filas) == 1 and str(int(actual["precio_lista"])) == c["valor_nuevo"]
              and str(int(actual["precio_lista_anterior"])) == c["valor_anterior"])
    elif c["tratamiento"] == "SCD1":
        ok = len(filas) == 1                     # se sobrescribe: no se crea una versión nueva
        if c["atributo"] == "estado":
            ok = ok and actual["estado"] == c["valor_nuevo"]
    else:                                         # Inserción
        ok = len(filas) == 1 and actual["fecha_inicio_vigencia"] == config.FECHA_INICIO_POR_LOTE[2]
    resultados.append({"maestro": c["maestro"], "tratamiento": c["tratamiento"], "ok": ok})

resultados = pd.DataFrame(resultados)
resumen_scd = resultados.groupby(["maestro", "tratamiento"]).agg(esperados=("ok", "size"), verificados=("ok", "sum")).reset_index()
assert resultados["ok"].all(), "Hay cambios SCD que no se aplicaron como se esperaba"
logger.info(f"SCD verificado: {resultados['ok'].sum()} de {len(resultados)} cambios correctos")
resumen_scd

2026-10-26 04:44:09 | INFO    | gold | SCD verificado: 80 de 80 cambios correctos


,maestro,tratamiento,esperados,verificados
0,clientes,Inserción,30,30
1,clientes,SCD1,5,5
2,clientes,SCD2,35,35
3,productos,Inserción,3,3
4,productos,SCD1,1,1
5,productos,SCD2,2,2
6,productos,SCD3,4,4


### Ejemplo: un cliente que se mudó de comuna

Sus ventas del lote 1 quedan asociadas a la versión antigua (comuna anterior) y las del lote 2 a la nueva.

In [12]:
ejemplo = cambios[(cambios["maestro"] == "clientes") & (cambios["atributo"] == "comuna")].iloc[0]
id_seud = seud(ejemplo["registro"])

versiones = pd.read_sql(text("""
    SELECT c.sk_cliente, c.comuna, c.region, c.segmento, c.fecha_inicio_vigencia, c.fecha_fin_vigencia,
           c.es_actual, COUNT(f.sk_cliente) AS ventas_asociadas
    FROM dim_cliente c LEFT JOIN fact_ventas f ON f.sk_cliente = c.sk_cliente
    WHERE c.id_cliente_seud = :id
    GROUP BY c.sk_cliente ORDER BY c.fecha_inicio_vigencia"""), motor, params={"id": id_seud})
logger.info(f"=== Fin capa Gold ===")
versiones

2026-10-26 04:44:09 | INFO    | gold | === Fin capa Gold ===


,sk_cliente,comuna,region,segmento,fecha_inicio_vigencia,fecha_fin_vigencia,es_actual,ventas_asociadas
0,175,Maipú,Metropolitana,Nuevo,1900-01-01,2026-06-30,0,20
1,417,Viña del Mar,Valparaíso,Nuevo,2026-07-01,9999-12-31,1,6
